In [0]:
import pandas as pd

dbutils.widgets.text("master_data_file", "/Volumes/workspace/hdb/hdb-output-data/hdb_master_data.csv")

master_data_file = dbutils.widgets.get("master_data_file")

df = pd.read_csv(master_data_file)

In [0]:
df['town'].unique()

In [0]:
df['flat_type'].unique()

In [0]:
df['flat_model'].unique()


In [0]:

df['block'].unique()


In [0]:

# To find anomolies in relase price comparing resale_price against floor_area_sqm in same kind of flat (in same town, same year, same flat model)

# Calculate the unit price per sqm
df["price_per_sqm"] = df["resale_price"] / df["floor_area_sqm"]

# IQR Outlier Detection for groups (town, year, flat_model)
def detect_iqr_outliers(group, factor=1.5):
  q1 = group["price_per_sqm"].quantile(0.25)
  q3 = group["price_per_sqm"].quantile(0.75)
  iqr = q3 - q1
  lower_bound = q1 - (factor * iqr)
  upper_bound = q3 + (factor * iqr)
  return (group["price_per_sqm"] < lower_bound) | (
      group["price_per_sqm"] > upper_bound
  )


# Apply within flat groups (town, year, flat_model)
df["year"] = df["month"].astype(str).str[:4]
df["is_anomaly_iqr"] = df.groupby(["town", "year", "flat_model"]).apply(
    detect_iqr_outliers
).values

anomalies = df[df['is_anomaly_iqr'] == True]
anomalies.head(10)


In [0]:
counts = df['is_anomaly_iqr'].value_counts()
percentages = df['is_anomaly_iqr'].value_counts(normalize=True) * 100

summary = pd.DataFrame(
    {'Count': counts, 'Percentage (%)': percentages.round(2)}
)
summary.index = summary.index.map(
    {False: 'Normal', True: 'Anomaly'}
)

summary.head()